# jax-dpo-min on Colab TPU

Runs the Gemma-2B-IT recipe on a Colab TPU runtime (v5e-1 or v6e-1 single-core). bf16 is used for throughput, not memory — v5e has 16 GB HBM and v6e has 32 GB, so Gemma-2B fp32 (~8 GB) would fit, but bf16 matmul is meaningfully faster on these chips. Single-device; no pmap or sharding (multi-device is an explicit non-goal per `background.md`).

**Runtime type: TPU.** Pick this from Runtime → Change runtime type before running the first cell.

## 1. Install JAX with TPU backend.

In [ ]:
!pip install -q -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q -U flax optax orbax-checkpoint transformers datasets sentencepiece pyyaml

import jax
print('devices:', jax.devices())
assert jax.devices()[0].platform == 'tpu', 'No TPU detected — set Runtime → Change runtime type → TPU.'

## 2. Mount Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo.

In [ ]:
%cd /content
![ -d jax-dpo-min ] || git clone https://github.com/jman4162/jax-dpo-min.git
%cd jax-dpo-min

## 4. (Optional) HuggingFace auth if the model is gated.

In [ ]:
# from huggingface_hub import login; login()

## 5. Train.

`gemma_tpu.yaml` sets `dtype: bfloat16`. On v6e-1 (32 GB) you can comfortably bump `batch_size` to 8; on v5e-1 (16 GB), keep it at 4.

In [ ]:
!python train.py --config configs/gemma_tpu.yaml --output-dir /content/drive/MyDrive/jax-dpo-min/runs/gemma_tpu_run01

## 6. Evaluate.

In [ ]:
!python eval.py --checkpoint /content/drive/MyDrive/jax-dpo-min/runs/gemma_tpu_run01/step_001500